# SurfaceTarget-CRC — phase 06 — patch 6 1

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 6 — PATCH 6.1
#
# RESEARCH FIGURES AND FINAL EVIDENCE VERIFICATION
#
# OBJECTIVES:
# 1. Validate completed Phase 5 research outputs
# 2. Verify the 15-candidate panel and 948-gene universe
# 3. Reconfirm cross-scenario ranking membership
# 4. Generate evidence-coverage figure
# 5. Generate protein-topology distribution figure
# 6. Generate surface-candidate ranking-sensitivity figure
# 7. Prepare figure-source manifest
# 8. Save a preliminary Results evidence audit
# 9. Save provenance
# 10. Create a Phase 6 research checkpoint
#
# IMPORTANT:
# - No new therapeutic rankings
# - No candidate exclusions
# - No clinical safety assessments
# - No therapeutic efficacy claims
# - No source data overwritten
# - All figures are descriptive
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    RESULTS,
    FIGURES,
    META,
    REPORTS,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

ROADMAP_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)

INTEGRITY_PATH = (
    META /
    "surfacetarget_phase5_roundtrip_integrity_audit.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 6 — PATCH 6.1")
print("RESEARCH FIGURES AND EVIDENCE VERIFICATION")
print("=" * 65)


# ------------------------------------------------------------
# 3. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PHASE 5 RESEARCH OUTPUTS")
print("-" * 65)


required_files = {
    "Research panel": PANEL_PATH,
    "Complete gene universe": FULL_PATH,
    "Experimental roadmap": ROADMAP_PATH,
    "Ranking correlations": CORRELATION_PATH,
    "Panel provenance": PANEL_PROVENANCE_PATH,
    "Integrity audit": INTEGRITY_PATH
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )

    print(f"{name}: FOUND")


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PANEL_PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


with open(
    INTEGRITY_PATH,
    "r"
) as handle:

    integrity = json.load(handle)


if not integrity.get("integrity_passed", False):

    raise RuntimeError(
        "Previous scientific integrity audit did not pass."
    )


if integrity.get(
    "substantive_discrepancies",
    []
):

    raise RuntimeError(
        "Previous audit contains substantive discrepancies."
    )


if sha256_file(PANEL_PATH) != previous_provenance.get(
    "panel_sha256"
):

    raise RuntimeError(
        "Research panel checksum mismatch."
    )


if sha256_file(FULL_PATH) != previous_provenance.get(
    "full_audit_sha256"
):

    raise RuntimeError(
        "Full candidate audit checksum mismatch."
    )


print("\nPrevious scientific integrity audit: VERIFIED")

print("Research panel checksum: PASSED")

print("Complete universe checksum: PASSED")


# ------------------------------------------------------------
# 4. LOAD RESEARCH TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING RESEARCH EVIDENCE")
print("-" * 65)


panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


full = pd.read_csv(
    FULL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


roadmap = pd.read_csv(
    ROADMAP_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


if len(full) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if len(panel) != 15:

    raise RuntimeError(
        "Expected 15 research-panel genes."
    )


if len(roadmap) != 15:

    raise RuntimeError(
        "Expected 15 experimental validation plans."
    )


for name, table in {
    "Full": full,
    "Panel": panel,
    "Roadmap": roadmap
}.items():

    if "gene_id" not in table.columns:

        raise RuntimeError(
            f"{name}: missing gene_id."
        )

    if table["gene_id"].isna().any():

        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


if not set(panel["gene_id"]).issubset(
    set(full["gene_id"])
):

    raise RuntimeError(
        "Research panel contains unexpected genes."
    )


if set(panel["gene_id"]) != set(
    roadmap["gene_id"]
):

    raise RuntimeError(
        "Experimental roadmap does not match panel."
    )


print("\nFull candidate universe:", len(full))

print("Research panel:", len(panel))

print("Experimental validation plans:", len(roadmap))

print("\nGene identity validation: PASSED")


# ------------------------------------------------------------
# 5. STRICT BOOLEAN HELPER
# ------------------------------------------------------------

def as_bool(series, name):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true",
        "false",
        "1",
        "0",
        "",
        "nan",
        "none",
        "<na>"
    }

    invalid = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if invalid.any():

        raise RuntimeError(
            f"Unexpected boolean values in {name}: "
            f"{values[invalid].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


# ------------------------------------------------------------
# 6. VERIFY EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING CROSS-OMICS EVIDENCE COVERAGE")
print("-" * 65)


required_coverage_columns = {
    "protein_enriched",
    "uniprot_reviewed_available",
    "hpa_subcellular_record_available",
    "phase5_normal_ihc_available",
    "depmap_final_mapping_available",
    "depmap_crc_models_measured",
    "phase5_stronger_surface_annotation",
    "phase5_topology_class",
    "phase5_followup_lane",
    "phase56_in_review_panel"
}


missing = required_coverage_columns - set(full.columns)

if missing:

    raise RuntimeError(
        f"Missing evidence fields: {sorted(missing)}"
    )


protein_supported = as_bool(
    full["protein_enriched"],
    "protein_enriched"
)


uniprot_reviewed = as_bool(
    full["uniprot_reviewed_available"],
    "uniprot_reviewed_available"
)


hpa_subcellular = as_bool(
    full["hpa_subcellular_record_available"],
    "hpa_subcellular_record_available"
)


normal_ihc = as_bool(
    full["phase5_normal_ihc_available"],
    "phase5_normal_ihc_available"
)


depmap_mapped = as_bool(
    full["depmap_final_mapping_available"],
    "depmap_final_mapping_available"
)


stronger_surface = as_bool(
    full["phase5_stronger_surface_annotation"],
    "phase5_stronger_surface_annotation"
)


panel_membership = as_bool(
    full["phase56_in_review_panel"],
    "phase56_in_review_panel"
)


depmap_measured = pd.to_numeric(
    full["depmap_crc_models_measured"],
    errors="raise"
)


coverage_data = {

    "Research candidates": len(full),

    "UniProt reviewed mappings":
        int(uniprot_reviewed.sum()),

    "DepMap verified mappings":
        int(depmap_mapped.sum()),

    "DepMap >=80% CRC coverage":
        int(depmap_measured.ge(65).sum()),

    "HPA subcellular annotations":
        int(hpa_subcellular.sum()),

    "HPA normal-tissue IHC":
        int(normal_ihc.sum()),

    "Stronger surface-related topology":
        int(stronger_surface.sum()),

    "Qualifying CPTAC enrichment":
        int(protein_supported.sum())
}


expected_counts = {

    "Research candidates": 948,

    "UniProt reviewed mappings": 944,

    "DepMap verified mappings": 943,

    "DepMap >=80% CRC coverage": 923,

    "HPA subcellular annotations": 755,

    "HPA normal-tissue IHC": 690,

    "Stronger surface-related topology": 112,

    "Qualifying CPTAC enrichment": 20
}


if coverage_data != expected_counts:

    raise RuntimeError(
        "Evidence counts differ from previously "
        "validated results:\n"
        f"Observed: {coverage_data}\n"
        f"Expected: {expected_counts}"
    )


if int(panel_membership.sum()) != 15:

    raise RuntimeError(
        "Expected exactly 15 research-panel members."
    )


print("\nEvidence coverage:")

for name, count in coverage_data.items():

    print(f"{name}: {count}")


print("\nCoverage audit: PASSED")

print(
    "These categories overlap and are not "
    "sequential filtering stages."
)


# ------------------------------------------------------------
# 7. VALIDATE PROTEIN TOPOLOGY CLASSES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PROTEIN TOPOLOGY DISTRIBUTION")
print("-" * 65)


topology_counts = (
    full["phase5_topology_class"]
    .value_counts()
)


expected_topology = {

    "No qualifying surface-topology evidence "
    "in extracted annotations": 552,

    "Secreted/extracellular; "
    "membrane attachment not established": 128,

    "Membrane with annotated extracellular region": 104,

    "Transmembrane; surface localization unresolved": 88,

    "Membrane-associated; "
    "extracellular topology unresolved": 64,

    "GPI-anchored membrane-associated": 8,

    "Other or insufficient topology annotation": 4
}


if topology_counts.to_dict() != expected_topology:

    raise RuntimeError(
        "Topology distribution differs from Patch 5.2."
    )


print("\n")

print(topology_counts.to_string())

print("\nTopology classification: VERIFIED")


# ------------------------------------------------------------
# 8. VALIDATE CROSS-SCENARIO CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING SENSITIVITY")
print("-" * 65)


SCENARIO_COLUMNS = {

    "Protein-first":
        "phase5_rank_protein_first",

    "Normal-review-first":
        "phase5_rank_normal_review_first",

    "RNA-discovery-first":
        "phase5_rank_rna_discovery_first"
}


surface = full[
    full["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 putative surface candidates."
    )


for column in SCENARIO_COLUMNS.values():

    values = pd.to_numeric(
        surface[column],
        errors="raise"
    )

    if values.isna().any():

        raise RuntimeError(
            f"Missing scenario ranks: {column}"
        )

    if set(values.astype(int)) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Unexpected rank distribution: {column}"
        )


three_way_mask = (
    surface[
        list(SCENARIO_COLUMNS.values())
    ]
    .le(20)
    .all(axis=1)
)


three_way = surface[
    three_way_mask
].copy()


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(three_way["gene_name"]) != expected_three:

    raise RuntimeError(
        "Three-scenario candidate identities changed."
    )


print("\nThree-scenario top-20 candidates:")

print(
    three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS.values()
        ]
    ].to_string(index=False)
)


print(
    "\nThree-scenario agreement: VERIFIED"
)


# ------------------------------------------------------------
# 9. FIGURE 1 — EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 1 — EVIDENCE COVERAGE")
print("-" * 65)


fig1_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig1_Evidence_Coverage.png"
)


labels = list(coverage_data.keys())

values = list(coverage_data.values())


fig, ax = plt.subplots(
    figsize=(12, 7)
)


positions = np.arange(
    len(labels)
)


bars = ax.barh(
    positions,
    values
)


ax.set_yticks(positions)

ax.set_yticklabels(labels)

ax.invert_yaxis()


ax.set_xlim(
    0,
    1050
)


ax.set_xlabel(
    "Candidate genes"
)


ax.set_title(
    "SurfaceTarget-CRC — Cross-Omics Evidence Coverage",
    fontsize=15,
    pad=18
)


for bar, value in zip(bars, values):

    ax.text(
        value + 12,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10
    )


ax.text(
    0,
    -0.13,
    "",
    transform=ax.transAxes
)


fig.text(
    0.5,
    0.01,
    "Evidence categories overlap; bars do not represent a sequential screening funnel.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig1_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 1 saved:", fig1_path)


# ------------------------------------------------------------
# 10. FIGURE 2 — PROTEIN TOPOLOGY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 2 — PROTEIN TOPOLOGY")
print("-" * 65)


fig2_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig2_Topology_Distribution.png"
)


topology_display_labels = {

    "No qualifying surface-topology evidence "
    "in extracted annotations":
        "No qualifying surface topology",

    "Secreted/extracellular; "
    "membrane attachment not established":
        "Secreted / extracellular",

    "Membrane with annotated extracellular region":
        "Membrane + extracellular region",

    "Transmembrane; surface localization unresolved":
        "Transmembrane; residency unresolved",

    "Membrane-associated; "
    "extracellular topology unresolved":
        "Membrane; extracellular topology unresolved",

    "GPI-anchored membrane-associated":
        "GPI-anchored membrane",

    "Other or insufficient topology annotation":
        "Other / insufficient annotation"
}


plot_topology = (
    topology_counts
    .rename(index=topology_display_labels)
)


fig, ax = plt.subplots(
    figsize=(12, 7)
)


bars = ax.barh(
    plot_topology.index,
    plot_topology.values
)


ax.invert_yaxis()


ax.set_xlabel(
    "Candidate genes"
)


ax.set_title(
    "SurfaceTarget-CRC — Protein Topology Evidence",
    fontsize=15,
    pad=18
)


ax.set_xlim(
    0,
    max(plot_topology.values) * 1.15
)


for bar, value in zip(
    bars,
    plot_topology.values
):

    ax.text(
        value + 5,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10
    )


fig.text(
    0.5,
    0.01,
    "Topology annotations do not independently establish antibody accessibility.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig2_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 2 saved:", fig2_path)


# ------------------------------------------------------------
# 11. FIGURE 3 — RANKING SENSITIVITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 3 — RANKING SENSITIVITY")
print("-" * 65)


fig3_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png"
)


# Only surface-antigen research groups A and B
# are comparable in this figure.
#
# FAP, membrane-accessibility and extracellular
# candidates are excluded from this visual
# because they belong to different
# biological follow-up categories.


surface_panel = panel[
    panel["phase56_review_group"]
    .str.startswith(("A", "B"))
].copy()


if len(surface_panel) != 9:

    raise RuntimeError(
        "Expected nine surface-panel genes."
    )


surface_panel = surface_panel.sort_values(
    [
        "phase56_review_group",
        "phase56_group_display_order"
    ]
)


plot_positions = np.arange(
    len(surface_panel)
)


fig, ax = plt.subplots(
    figsize=(12, 7)
)


markers = ["o", "s", "^"]

offsets = [-0.18, 0.0, 0.18]


for (label, column), marker, offset in zip(
    SCENARIO_COLUMNS.items(),
    markers,
    offsets
):

    ax.scatter(
        pd.to_numeric(
            surface_panel[column],
            errors="raise"
        ),
        plot_positions + offset,
        marker=marker,
        s=80,
        label=label
    )


ax.set_yticks(plot_positions)

ax.set_yticklabels(
    surface_panel["gene_name"]
)


ax.invert_yaxis()


ax.set_xlim(
    0,
    115
)


ax.set_xticks(
    [1, 20, 40, 60, 80, 100]
)


ax.axvline(
    20,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Rank within the 111-candidate surface-antigen lane"
)


ax.set_title(
    "SurfaceTarget-CRC — Prioritization Scenario Sensitivity",
    fontsize=15,
    pad=18
)


ax.legend(
    loc="lower right"
)


ax.grid(
    axis="x",
    alpha=0.2
)


fig.text(
    0.5,
    0.01,
    "Lower ranks indicate earlier research review, not greater therapeutic efficacy.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig3_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 3 saved:", fig3_path)


# ------------------------------------------------------------
# 12. VERIFY FIGURE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING GENERATED FIGURES")
print("-" * 65)


figure_paths = {
    "Figure 1 — Evidence coverage": fig1_path,
    "Figure 2 — Protein topology": fig2_path,
    "Figure 3 — Ranking sensitivity": fig3_path
}


for name, path in figure_paths.items():

    if not path.exists():

        raise RuntimeError(
            f"Figure was not saved: {name}"
        )

    if path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty figure file: {name}"
        )

    print(
        f"{name}: SAVED "
        f"({path.stat().st_size / 1024:.1f} KiB)"
    )


# ------------------------------------------------------------
# 13. PREPARE FIGURE-SOURCE MANIFEST
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING FIGURE-SOURCE MANIFEST")
print("-" * 65)


manifest_records = [

    {
        "figure": "Figure 1",
        "title": "Cross-Omics Evidence Coverage",
        "path": str(fig1_path),
        "source": str(FULL_PATH),
        "description": (
            "Counts of overlapping evidence "
            "categories within 948 research genes"
        )
    },

    {
        "figure": "Figure 2",
        "title": "Protein Topology Evidence",
        "path": str(fig2_path),
        "source": str(FULL_PATH),
        "description": (
            "Distribution of Phase 5 protein "
            "topology annotations"
        )
    },

    {
        "figure": "Figure 3",
        "title": "Prioritization Scenario Sensitivity",
        "path": str(fig3_path),
        "source": str(PANEL_PATH),
        "description": (
            "Three within-lane ranking scenarios "
            "for nine surface-antigen review genes"
        )
    }

]


manifest = pd.DataFrame(
    manifest_records
)


manifest["figure_sha256"] = (
    manifest["path"]
    .map(
        lambda value: sha256_file(
            Path(value)
        )
    )
)


MANIFEST_PATH = (
    RESULTS /
    "surfacetarget_phase6_figure_manifest.csv"
)


manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


print("\nFigure manifest saved.")


# ------------------------------------------------------------
# 14. CREATE PRELIMINARY RESULTS AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PRELIMINARY RESULTS AUDIT")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 6.1 — Evidence and Figure Audit",

    "",

    (
        "This document records validated "
        "research evidence used for the "
        "initial SurfaceTarget-CRC figures."
    ),

    "",

    (
        "The analyses concern exploratory "
        "therapeutic targeting hypotheses. "
        "They do not demonstrate therapeutic "
        "efficacy or clinical safety."
    ),

    "",

    "## Candidate inventory",

    "",

    f"- Original candidate universe: {len(full)} genes.",

    f"- Final research-review panel: {len(panel)} genes.",

    f"- Stronger surface-related annotations: "
    f"{int(stronger_surface.sum())} genes.",

    f"- Qualifying CPTAC protein enrichment: "
    f"{int(protein_supported.sum())} genes.",

    f"- Normal-tissue HPA IHC annotations: "
    f"{int(normal_ihc.sum())} genes.",

    f"- Verified DepMap mappings: "
    f"{int(depmap_mapped.sum())} genes.",

    "",

    "## Figure 1 — Evidence coverage",

    "",

    (
        "Evidence availability varies "
        "substantially across molecular and "
        "protein-annotation resources."
    ),

    (
        "These categories overlap and must "
        "not be presented as sequential "
        "candidate elimination stages."
    ),

    "",

    (
        "![Evidence coverage]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)"
    ),

    "",

    "## Figure 2 — Protein topology",

    "",

    (
        "The topology figure separates "
        "membrane-associated proteins, "
        "secreted proteins, and candidates "
        "without qualifying surface annotations."
    ),

    (
        "These classifications are based "
        "on extracted biological annotations "
        "rather than direct antibody-binding assays."
    ),

    "",

    (
        "![Protein topology]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig2_Topology_Distribution.png)"
    ),

    "",

    "## Figure 3 — Ranking sensitivity",

    "",

    (
        "The surface-antigen research panel "
        "was evaluated under three different "
        "research-priority scenarios."
    ),

    (
        "Protein-first and normal-review-first "
        "ordering assumptions were highly "
        "correlated in the original analysis."
    ),

    (
        "Cross-scenario agreement does not "
        "constitute independent biological "
        "validation."
    ),

    "",

    (
        "![Ranking sensitivity]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)"
    ),

    "",

    "## Three-scenario-consistent candidates",

    ""
]


for gene in sorted(expected_three):

    report_lines.append(
        f"- {gene}"
    )


report_lines.extend([

    "",

    (
        "These three genes appeared in "
        "the top 20 of all three scenarios "
        "within the putative surface-antigen "
        "review lane."
    ),

    "",

    (
        "This result reflects ranking "
        "consistency under the selected "
        "rules, not therapeutic superiority."
    ),

    "",

    "## Scope and limitations",

    "",

    (
        "The TCGA discovery workflow "
        "included computationally low-MSI "
        "samples that are not independently "
        "confirmed MSS cases."
    ),

    "",

    (
        "The CPTAC enrichment analysis "
        "used an independent paired MSS "
        "colon cancer proteomic cohort."
    ),

    "",

    (
        "The DepMap colorectal cell-line "
        "cohort has not been established "
        "as entirely MSS."
    ),

    "",

    (
        "Bulk tumor RNA and proteomics "
        "cannot independently determine "
        "whether a signal originates from "
        "malignant epithelial cells or "
        "the tumor microenvironment."
    ),

    "",

    (
        "Normal-tissue IHC findings are "
        "descriptive and do not predict "
        "clinical toxicity."
    ),

    "",

    (
        "Protein topology annotations "
        "do not prove extracellular epitope "
        "accessibility or antibody "
        "internalization."
    ),

    "",

    (
        "CRISPR gene-effect values describe "
        "cell fitness after gene knockout; "
        "they do not directly predict "
        "antibody-drug conjugate efficacy."
    ),

    "",

    (
        "All 948 genes remain retained "
        "in the full research dataset."
    ),

    "",

    (
        "No therapeutic target has been "
        "experimentally validated by this project."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nEvidence audit report saved.")

print("Location:", REPORT_PATH)


# ------------------------------------------------------------
# 15. SAVE PHASE 6 PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 6 PROVENANCE")
print("-" * 65)


PROVENANCE_OUTPUT = (
    META /
    "surfacetarget_phase6_figure_provenance.json"
)


phase6_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "6.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "candidate_universe": 948,

    "research_panel_size": 15,

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)

        for name, path in required_files.items()

    },

    "evidence_coverage": coverage_data,

    "topology_distribution": (
        topology_counts.to_dict()
    ),

    "three_scenario_consistent_genes":
        sorted(expected_three),

    "figures": {

        name: {

            "path": str(path),

            "sha256": sha256_file(path)

        }

        for name, path in figure_paths.items()

    },

    "figure_manifest": str(
        MANIFEST_PATH
    ),

    "figure_manifest_sha256":
        sha256_file(MANIFEST_PATH),

    "report": str(
        REPORT_PATH
    ),

    "report_sha256":
        sha256_file(REPORT_PATH),

    "source_evidence_modified": False,

    "new_candidate_ranking_performed": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "limitations": [

        "Evidence coverage categories overlap.",

        "Protein topology is based on "
        "annotations rather than binding assays.",

        "Prioritization scenarios are "
        "not independent experiments.",

        "Normal-tissue expression does "
        "not establish therapeutic safety.",

        "Bulk tumor measurements do "
        "not establish malignant-cell origin.",

        "The TCGA discovery cohort is "
        "not uniformly confirmed MSS.",

        "The DepMap colorectal cohort "
        "is not MSS-confirmed here.",

        "No therapeutic efficacy "
        "has been demonstrated."

    ]
}


with open(
    PROVENANCE_OUTPUT,
    "w"
) as handle:

    json.dump(
        phase6_provenance,
        handle,
        indent=4
    )


print("\nFigure provenance saved.")


# ------------------------------------------------------------
# 16. CREATE PHASE 6 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 6 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase6_Patch61_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 6.1 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(full))

print("Final research panel:", len(panel))

print(
    "Stronger surface-related annotations:",
    int(stronger_surface.sum())
)


print(
    "Qualifying CPTAC-supported candidates:",
    int(protein_supported.sum())
)


print(
    "Three-scenario-consistent surface genes:",
    len(three_way)
)


print("\nFigures created:")

for name in figure_paths:

    print("-", name)


print("\nSaved research outputs:")

print("1. SurfaceTarget_CRC_Fig1_Evidence_Coverage.png")

print("2. SurfaceTarget_CRC_Fig2_Topology_Distribution.png")

print("3. SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png")

print("4. surfacetarget_phase6_figure_manifest.csv")

print("5. SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md")

print("6. surfacetarget_phase6_figure_provenance.json")


print("\nOriginal research evidence: PRESERVED")

print("Original candidate rankings: UNCHANGED")

print("No candidate genes excluded.")

print("No therapeutic efficacy scores assigned.")

print("No clinical safety predictions made.")


print("\nPhase 6 checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 6.1 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC — PHASE 6, PATCH 6.2
# EVIDENCE-GROUNDED FORMAL RESEARCH REPORT DRAFT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

ROOT = Path('/content/SurfaceTarget-CRC')

TABLES = ROOT / 'results/tables'
FIGURES = ROOT / 'results/figures'
META = ROOT / 'data/metadata'
REPORTS = ROOT / 'reports'
CHECKPOINTS = ROOT / 'checkpoints'

for directory in [
    TABLES,
    FIGURES,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(parents=True, exist_ok=True)


PANEL_PATH = (
    TABLES /
    'surfacetarget_phase5_research_panel_v1.csv'
)

FULL_PATH = (
    TABLES /
    'surfacetarget_phase5_full_universe_panel_audit.csv'
)

ROADMAP_PATH = (
    TABLES /
    'surfacetarget_phase5_experimental_validation_roadmap.csv'
)

CORRELATION_PATH = (
    TABLES /
    'surfacetarget_phase5_scenario_rank_correlations.csv'
)

MANIFEST_PATH = (
    TABLES /
    'surfacetarget_phase6_figure_manifest.csv'
)

PANEL_PROVENANCE_PATH = (
    META /
    'surfacetarget_phase5_research_panel_provenance.json'
)

FIGURE_PROVENANCE_PATH = (
    META /
    'surfacetarget_phase6_figure_provenance.json'
)


print('=' * 68)
print('SURFACETARGET-CRC | PHASE 6 — PATCH 6.2')
print('EVIDENCE-GROUNDED FORMAL RESEARCH REPORT DRAFT')
print('=' * 68)


# ============================================================
# 2. VALIDATION HELPERS
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b''
        ):
            digest.update(chunk)

    return digest.hexdigest()


def require(condition, message):

    if not condition:
        raise RuntimeError(message)


def read_json(path):

    with open(path, encoding='utf-8') as handle:
        return json.load(handle)


def to_bool(series):

    values = (
        series.astype('string')
        .str.strip()
        .str.lower()
    )

    valid = {
        'true', 'false', '1', '0',
        '', 'nan', 'none', '<na>'
    }

    invalid = (
        values.notna()
        & ~values.isin(valid)
    )

    require(
        not invalid.any(),
        f'Invalid boolean tokens: '
        f'{values[invalid].unique().tolist()}'
    )

    return (
        values.isin(['true', '1'])
        .fillna(False)
        .astype(bool)
    )


def md(value):

    if pd.isna(value):
        return 'Not available'

    return (
        str(value)
        .replace('|', r'\|')
        .replace('\n', ' ')
        .strip()
    )


def fmt(value, digits=3):

    num = pd.to_numeric(
        pd.Series([value]),
        errors='coerce'
    ).iloc[0]

    return (
        f'{num:.{digits}f}'
        if pd.notna(num) and np.isfinite(num)
        else 'Not available'
    )


# ============================================================
# 3. VERIFY RESEARCH INPUTS
# ============================================================

print('\nCHECKING INPUTS AND CHECKSUMS')


files = {

    'Panel': PANEL_PATH,

    '948-gene audit': FULL_PATH,

    'Validation roadmap': ROADMAP_PATH,

    'Rank correlations': CORRELATION_PATH,

    'Figure manifest': MANIFEST_PATH,

    'Panel provenance': PANEL_PROVENANCE_PATH,

    'Figure provenance': FIGURE_PROVENANCE_PATH

}


for name, path in files.items():

    require(
        path.is_file(),
        f'Missing {name}: {path}'
    )


p56 = read_json(PANEL_PROVENANCE_PATH)

p61 = read_json(FIGURE_PROVENANCE_PATH)


require(
    sha256_file(PANEL_PATH) == p56.get('panel_sha256'),
    'Panel SHA256 mismatch'
)

require(
    sha256_file(FULL_PATH) == p56.get('full_audit_sha256'),
    'Full-audit SHA256 mismatch'
)

require(
    sha256_file(MANIFEST_PATH)
    == p61.get('figure_manifest_sha256'),
    'Figure-manifest SHA256 mismatch'
)


# Verify saved Figure 6.1 PNGs.

for label, info in p61.get('figures', {}).items():

    path = Path(info['path'])

    require(
        path.is_file(),
        f'Missing figure {label}: {path}'
    )

    require(
        sha256_file(path) == info['sha256'],
        f'Figure checksum mismatch: {label}'
    )


require(
    len(p61.get('figures', {})) == 3,
    'Expected three validated figures'
)


# ============================================================
# 4. LOAD VALIDATED RESEARCH TABLES
# ============================================================

print('\nLOADING VALIDATED RESEARCH TABLES')


panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        'gene_id': str,
        'ensembl_stable_id': str
    },
    low_memory=False
)


full = pd.read_csv(
    FULL_PATH,
    dtype={
        'gene_id': str,
        'ensembl_stable_id': str
    },
    low_memory=False
)


roadmap = pd.read_csv(
    ROADMAP_PATH,
    dtype={'gene_id': str},
    low_memory=False
)


corr = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


manifest = pd.read_csv(
    MANIFEST_PATH
)


for name, frame, expected in [

    ('panel', panel, 15),

    ('full', full, 948),

    ('roadmap', roadmap, 15)

]:

    require(
        len(frame) == expected,
        f'Unexpected {name} row count'
    )

    require(
        'gene_id' in frame.columns,
        f'Missing gene_id in {name}'
    )

    require(
        frame['gene_id'].notna().all()
        and frame['gene_id'].is_unique,
        f'Invalid gene IDs in {name}'
    )


require(
    set(panel.gene_id) == set(roadmap.gene_id),
    'Panel/roadmap gene IDs differ'
)

require(
    set(panel.gene_id).issubset(set(full.gene_id)),
    'Panel missing from full universe'
)

require(
    to_bool(full['phase56_in_review_panel']).sum() == 15,
    'Incorrect full-audit panel flags'
)

require(
    set(panel.gene_id)
    ==
    set(
        full.loc[
            to_bool(full['phase56_in_review_panel']),
            'gene_id'
        ]
    ),
    'Panel membership disagreement'
)


# ============================================================
# 5. VALIDATE EVIDENCE FIELDS
# ============================================================

required_fields = [

    'gene_name',

    'phase56_review_group',

    'phase5_followup_lane',

    'phase5_rna_conservative_log2fc',

    'phase5_protein_evidence_state',

    'phase5_topology_class',

    'phase55_normal_tissues_medium_high',

    'depmap_crc_median_gene_effect',

    'mss_complete_pairs',

    'phase5_rank_protein_first',

    'phase5_rank_normal_review_first',

    'phase5_rank_rna_discovery_first'

]


require(
    all(col in panel.columns for col in required_fields),
    'Research panel missing required columns'
)


required_full_fields = [

    'protein_enriched',

    'uniprot_reviewed_available',

    'hpa_subcellular_record_available',

    'phase5_normal_ihc_available',

    'depmap_final_mapping_available',

    'depmap_crc_models_measured',

    'phase5_stronger_surface_annotation',

    'phase5_topology_class',

    'phase5_followup_lane'

]


require(
    all(col in full.columns for col in required_full_fields),
    'Full universe missing expected evidence'
)


# ============================================================
# 6. VALIDATE CROSS-OMICS EVIDENCE COUNTS
# ============================================================

print('\nVALIDATING CROSS-OMICS EVIDENCE COVERAGE')


coverage = {

    'Candidate universe':
        len(full),

    'Reviewed UniProt mappings':
        int(
            to_bool(
                full['uniprot_reviewed_available']
            ).sum()
        ),

    'Verified DepMap mappings':
        int(
            to_bool(
                full['depmap_final_mapping_available']
            ).sum()
        ),

    'High-coverage DepMap genes (>=65/81 models)':
        int(
            pd.to_numeric(
                full['depmap_crc_models_measured'],
                errors='coerce'
            ).ge(65).sum()
        ),

    'HPA subcellular annotations':
        int(
            to_bool(
                full['hpa_subcellular_record_available']
            ).sum()
        ),

    'HPA normal-tissue IHC':
        int(
            to_bool(
                full['phase5_normal_ihc_available']
            ).sum()
        ),

    'Stronger surface-related annotation':
        int(
            to_bool(
                full['phase5_stronger_surface_annotation']
            ).sum()
        ),

    'Qualifying CPTAC enrichment':
        int(
            to_bool(
                full['protein_enriched']
            ).sum()
        )

}


expected = [
    948,
    944,
    943,
    923,
    755,
    690,
    112,
    20
]


require(
    list(coverage.values()) == expected,
    f'Unexpected evidence counts: {coverage}'
)


require(
    len(manifest) == 3,
    'Expected three figure records'
)


print('\nEvidence coverage:')

for name, count in coverage.items():
    print(f'{name}: {count}')


# ============================================================
# 7. VERIFY RANKING ROBUSTNESS
# ============================================================

print('\nVALIDATING RANKING ROBUSTNESS')


surface = full[
    full['phase5_followup_lane'].eq(
        'Putative cell-surface antigen review'
    )
].copy()


require(
    len(surface) == 111,
    'Expected 111 surface-review candidates'
)


ranks = [

    'phase5_rank_protein_first',

    'phase5_rank_normal_review_first',

    'phase5_rank_rna_discovery_first'

]


for rank in ranks:

    values = pd.to_numeric(
        surface[rank],
        errors='raise'
    )

    require(
        values.notna().all()
        and set(values.astype(int)) == set(range(1, 112)),
        f'Rank validity failed: {rank}'
    )


all_three = surface[
    surface[ranks].le(20).all(axis=1)
]


require(
    set(all_three['gene_name'])
    == {'CA9', 'ADAM12', 'NPC1L1'},
    'Three-way membership mismatch'
)


require(
    np.isclose(
        corr.loc[
            'protein_first',
            'normal_review_first'
        ],
        0.9860,
        atol=1e-3
    ),
    'Scenario correlation differs'
)


panel = panel.merge(

    roadmap[
        [
            'gene_id',
            'primary_question',
            'primary_assay',
            'orthogonal_assay',
            'additional_review'
        ]
    ],

    on='gene_id',

    how='left',

    validate='one_to_one'

)


require(
    panel['primary_assay'].notna().all(),
    'Missing validation plans'
)


print('\nChecksums and data identities: PASSED')

print(
    'Gene universe:',
    len(full),
    '| Panel:',
    len(panel),
    '| Figures:',
    len(manifest)
)

print(
    'Three-way top-20:',
    ', '.join(sorted(all_three['gene_name']))
)


# ============================================================
# 8. INITIALIZE SCIENTIFIC REPORT
# ============================================================

print('\nGENERATING STRUCTURED SCIENTIFIC REPORT')


report = []


def add(*lines):
    report.extend(lines)


add(

    '# SurfaceTarget-CRC',

    '## Multi-omics prioritization of colorectal cancer '
    'cell-surface and microenvironment target hypotheses',

    '',

    '**Technical research report — evidence-grounded '
    'draft (Phase 6.2)**',

    '',

    f'Generated: '
    f'{datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}',

    '',

    '> **Research-use only.** This is a computational '
    'target-hypothesis analysis. No candidate has been '
    'experimentally validated for antibody binding, '
    'therapeutic efficacy, or clinical safety.',

    '',

    '## Abstract',

    '',

    'Identifying potentially accessible tumor-associated '
    'targets requires more than differential RNA expression. '
    'SurfaceTarget-CRC integrates exploratory TCGA colorectal '
    'transcriptomic evidence, paired MSS colon proteomics from '
    'CPTAC, protein topology and normal-tissue observations '
    'from UniProt/HPA, and colorectal cell-line knockout fitness '
    'from DepMap. A total of **948 gene-level candidates** were '
    'retained from the transcriptomic workflow. Among them, '
    '**20** met the project’s qualifying paired-protein enrichment '
    'criteria, **112** had stronger surface-related topology '
    'annotations, and **943** were represented in the validated '
    'DepMap gene-effect mapping. Three deliberately different '
    'within-lane review scenarios were compared for **111 putative '
    'cell-surface candidates**. **CA9, ADAM12, and NPC1L1** occurred '
    'in all three top-20 lists, although the protein-first and '
    'normal-review-first rankings were strongly correlated '
    '(Spearman ρ = 0.986). The output is an explicitly selected, '
    'mechanism-stratified **15-gene research panel**, not an '
    'optimized or clinically validated target ranking.',

    '',

    '## 1. Background and objective',

    '',

    'Tumor/normal differential expression can identify research '
    'leads but does not establish plasma-membrane residency, '
    'extracellular epitope availability, malignant-cell specificity, '
    'favorable normal-tissue distribution, or drug-response effects. '
    'This project asks which colorectal tumor-associated genes '
    'warrant follow-up as potential cell-surface, stromal, or '
    'extracellular-microenvironment targeting hypotheses. '
    'It prioritizes auditability, class-specific interpretation, '
    'source-specific missingness, and explicit experimental '
    'validation needs.',

    '',

    '## 2. Datasets and methods',

    '',

    '### 2.1. Discovery transcriptomics — TCGA/GDC',

    '',

    'An earlier workflow performed colorectal tumor-versus-normal '
    'transcriptomic analyses, QC sensitivity checks, supporting '
    'expression review, and patient-paired directional assessment. '
    'The retained set contains **948** candidate genes. '
    'The discovery workflow included a **computationally low-MSI** '
    'subgroup that must **not** be described as independently '
    'confirmed microsatellite-stable (MSS). The same-study '
    'sensitivity analyses and matched-patient direction checks '
    'are supporting analyses, not independent external validation. '
    'For downstream descriptive ranking, a conservative effect-size '
    'field was defined as the minimum of baseline and QC sensitivity '
    'log2 fold changes.',

    '',

    '### 2.2. Paired protein evidence — CPTAC',

    '',

    'The paired MSS colon tumor/normal proteomics analysis used '
    'a within-patient Wilcoxon signed-rank framework and '
    'multiple-testing adjustment. Candidate-level interpretation '
    'distinguishes qualifying enrichment, adequately covered '
    'nonqualifying measurements, sparse paired observations, '
    'and unavailable coverage. **20/948** candidates met the '
    'project-defined enrichment requirements. Unavailable or '
    'sparse protein measurements were **not** interpreted as '
    'protein absence. The detailed paired-sample inclusion '
    'and significance settings remain in the earlier phase outputs.',

    '',

    '### 2.3. Localization and normal-tissue annotation '
    '— UniProt/HPA',

    '',

    'Reviewed UniProt annotation, membrane/extracellular topology, '
    'GPI anchoring, HPA subcellular localization, and normal-tissue '
    'IHC were preserved as distinct evidence dimensions. '
    'Seven mutually exclusive annotation classes were defined '
    'from extracted topology fields. HPA tissue-level IHC is '
    'descriptive; an unmeasured gene is not assigned zero '
    'normal-tissue staining. Tissue-level maxima do not '
    'resolve all cell types and are not quantitative '
    'clinical toxicity predictions.',

    '',

    '### 2.4. CRISPR functional context — DepMap 26Q3',

    '',

    'Ensembl-to-DepMap gene identity was validated through '
    'HGNC-approved Ensembl/NCBI Gene ID crosswalks. '
    '**943/948** candidates matched DepMap gene-effect columns; '
    '**923** were measured in at least **65 of 81** colorectal '
    'adenocarcinoma models. A **79-model** patient-deduplicated '
    'sensitivity cohort and a **1,260-model** heterogeneous '
    'non-bowel comparator were evaluated in Phase 4. '
    'No MSS status was inferred for the DepMap colorectal models. '
    'CRISPR knockout fitness is not a proxy for antibody binding, '
    'epitope accessibility, or antibody-drug conjugate efficacy.',

    '',

    '### 2.5. Biological lanes and ranking-sensitivity design',

    '',

    'Genes were divided into putative surface-antigen, stromal, '
    'membrane-accessibility, extracellular microenvironment, '
    'and alternative/topology-clarification follow-up lanes. '
    'The **111** putative surface-antigen candidates were ordered '
    'in three deterministic scenarios: protein-evidence-first, '
    'normal-tissue-review-first, and RNA-discovery-first. '
    'Rank comparisons are **within the same lane only**. '
    'These are lexicographic research-review rules, not weighted '
    'therapeutic-efficacy scores. The two scenarios based primarily '
    'on protein and normal-tissue annotation proved highly similar, '
    'so two-of-three top-20 agreement should not be treated '
    'as independent robustness.',

    '',

    '## 3. Results',

    '',

    '### 3.1. Evidence coverage among 948 candidates',

    '',

    '| Evidence category | Number of candidates |',

    '|---|---:|'

)


# ============================================================
# 9. RESULTS — EVIDENCE COVERAGE
# ============================================================

for label, number in coverage.items():

    add(
        f'| {md(label)} | {number} |'
    )


add(

    '',

    'The categories in this table **overlap** and do not '
    'represent successive eliminations from a funnel.',

    '',

    '![Figure 1 — Cross-omics evidence coverage]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)',

    '',

    '*Figure 1. Source-specific coverage among the same '
    '948 research genes. Categories overlap.*',

    '',

    '### 3.2. Topology classification',

    '',

    '| Extracted topology annotation class | Genes |',

    '|---|---:|'

)


for category, count in (
    full['phase5_topology_class']
    .value_counts()
    .items()
):

    add(
        f'| {md(category)} | {int(count)} |'
    )


add(

    '',

    'The **104** candidates with annotated extracellular '
    'membrane regions and **8** GPI-anchored membrane-associated '
    'candidates constitute the **112** with stronger surface-related '
    'annotations. Topology evidence alone does not demonstrate '
    'antibody recognition or cancer-cell localization.',

    '',

    '![Figure 2 — Protein topology evidence]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig2_Topology_Distribution.png)',

    '',

    '*Figure 2. Seven mutually exclusive annotation classes '
    'based on extracted UniProt features.*',

    '',

    '### 3.3. Ranking stability and its limitations',

    '',

    'Across the 111-gene cell-surface review lane, '
    'protein-first and normal-review-first rankings had '
    f'Spearman **ρ = '
    f'{corr.loc["protein_first", "normal_review_first"]:.3f}**, '
    'whereas protein-first versus RNA-first had '
    f'**ρ = '
    f'{corr.loc["protein_first", "rna_discovery_first"]:.3f}**. '
    'Only CA9, ADAM12, and NPC1L1 were ranked in the top 20 '
    'under all three scenarios. This is within-model agreement '
    'and does **not** constitute replicated biological confirmation.',

    '',

    '| Gene | Protein-first | Normal-review-first | RNA-first |',

    '|---|---:|---:|---:|'

)


for _, row in (
    all_three
    .sort_values('gene_name')
    .iterrows()
):

    add(

        f'| {md(row["gene_name"])} | '

        f'{int(row[ranks[0]])} | '

        f'{int(row[ranks[1]])} | '

        f'{int(row[ranks[2]])} |'

    )


add(

    '',

    '![Figure 3 — Scenario ranking sensitivity]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)',

    '',

    '*Figure 3. Within-lane priorities for the nine '
    'surface-antigen hypotheses in the panel; smaller '
    'ranks indicate earlier research review, not '
    'superior clinical efficacy.*',

    '',

    '### 3.4. Mechanism-stratified 15-gene research panel',

    '',

    'The panel consists of **3** cross-scenario surface '
    'hypotheses, **6** complementary surface-antigen '
    'hypotheses, **1** stromal candidate, **2** membrane-'
    'accessibility candidates, and **3** extracellular-'
    'microenvironment candidates. The selection is a '
    'transparent research-review decision; genes outside '
    'this panel remain in the complete 948-gene audit.',

    '',

    '| Group | Gene | Conservative RNA log2FC | CPTAC evidence | '
    'Normal tissues with medium/high IHC | '
    'Median CRC CRISPR effect |',

    '|---|---|---:|---|---:|---:|'

)


for _, row in panel.iterrows():

    group = (
        row['phase56_review_group']
        .split(' | ')[0]
    )

    add(

        f'| {group} | '

        f'{md(row["gene_name"])} | '

        f'{fmt(row["phase5_rna_conservative_log2fc"])} | '

        f'{md(row["phase5_protein_evidence_state"])} | '

        f'{fmt(row["phase55_normal_tissues_medium_high"], 0)} | '

        f'{fmt(row["depmap_crc_median_gene_effect"])} |'

    )


add(

    '',

    'The normal-tissue counts reflect HPA tissue categories '
    'with summarized medium/high IHC, not percentages of cells '
    'stained or a clinical safety margin. A missing value denotes '
    'unavailable IHC evidence. Positive CPTAC evidence must not '
    'be conflated with proven malignant-cell-specific expression.',

    '',

    '## 4. Candidate-specific interpretation and validation',

    ''

)


# ============================================================
# 10. CANDIDATE-SPECIFIC INTERPRETATION
# ============================================================

for group_name, group in panel.groupby(
    'phase56_review_group',
    sort=False
):

    add(
        f'### {md(group_name)}',
        ''
    )


    for _, row in group.iterrows():

        gene = md(
            row['gene_name']
        )

        normal_text = fmt(
            row['phase55_normal_tissues_medium_high'],
            0
        )


        add(

            f'**{gene}.** '

            f'Conservative RNA log2FC '
            f'{fmt(row["phase5_rna_conservative_log2fc"])}; '

            f'CPTAC: '
            f'{md(row["phase5_protein_evidence_state"])}; '

            f'topology: '
            f'{md(row["phase5_topology_class"])}; '

            f'normal medium/high IHC tissues: '
            f'{normal_text}; '

            f'CRC CRISPR median effect: '
            f'{fmt(row["depmap_crc_median_gene_effect"])}. '

            f'Primary question: '
            f'{md(row["primary_question"])} '

            f'Proposed initial assessment: '
            f'{md(row["primary_assay"])} '

            f'Orthogonal follow-up: '
            f'{md(row["orthogonal_assay"])} '

            f'Additional review: '
            f'{md(row["additional_review"])}',

            ''

        )


# ============================================================
# 11. DISCUSSION AND LIMITATIONS
# ============================================================

add(

    '## 5. Interpretation',

    '',

    'The evidence supports different **research pathways** '
    'rather than a single universal therapeutic target order. '
    'CA9, ADAM12, and NPC1L1 are consistent across chosen '
    'prioritization scenarios, yet none has qualifying CPTAC '
    'enrichment evidence under this workflow. CDH3 and CLDN1 '
    'have stronger RNA and extracellular-topology evidence but '
    'sparse paired proteomics and normal epithelial expression '
    'concerns. DPEP1 and SLC7A5 have qualifying protein enrichment '
    'but meaningful normal-tissue IHC observations. FAP is a '
    'distinct stromal targeting hypothesis; its gene knockout '
    'effect in isolated cancer models is not an informative '
    'measure of stromal-antigen utility. TDGF1 has GPI-associated '
    'topology and an appreciable DepMap fitness signal, but both '
    'normal-tissue IHC and paired CPTAC evidence are unavailable '
    'in this workflow. None of these findings identifies a '
    'favorable therapeutic index.',

    '',

    '## 6. Key limitations',

    '',

    '1. **MSS specificity:** The TCGA discovery subset includes '
    'computationally low-MSI cases; the colorectal DepMap cohort '
    'is not independently MSS-confirmed. Only the relevant CPTAC '
    'paired proteomics cohort is described as MSS-confirmed.',

    '2. **Bulk tumor origin:** RNA and protein enrichment can '
    'reflect malignant cells, stroma, vascular compartments, '
    'or inflammatory cells. Cellular origin is unresolved.',

    '3. **Surface accessibility:** Annotated extracellular domains '
    'or GPI anchors do not prove live-cell antibody binding, '
    'adequate antigen density, internalization, or therapeutic '
    'suitability.',

    '4. **Normal-tissue uncertainty:** HPA IHC may be missing or '
    'assay-dependent and is not a quantitative exposure or '
    'clinical toxicity measurement. Tissue-count rankings '
    'are coarse.',

    '5. **Proteomics missingness:** Sparse/unavailable paired '
    'protein coverage is not evidence of protein absence; '
    'adequately measured but nonqualifying genes require '
    'separate interpretation.',

    '6. **Screen model limitations:** DepMap reflects in-vitro '
    'knockout fitness and imperfect representation of primary '
    'tumors and their microenvironment.',

    '7. **Prioritization sensitivity:** Two scenarios are '
    'highly correlated, and the choice of lexicographic '
    'hierarchy can substantially change rank order.',

    '8. **No experimental confirmation:** No candidate received '
    'new wet-lab, binding, tumor-cell selectivity, or in-vivo '
    'efficacy validation in this project.',

    '',

    '## 7. Next experimental steps',

    '',

    'The accompanying '
    '`surfacetarget_phase5_experimental_validation_roadmap.csv` '
    'proposes gene-specific initial and orthogonal assays. '
    'A scientifically useful next study would confirm MSI/MSS '
    'status in selected models; localize expression to malignant '
    'versus nonmalignant compartments using spatial and single-cell '
    'evidence; measure live-cell surface binding with appropriately '
    'validated reagents; quantify antigen abundance relative to '
    'matched normal epithelia and organoids; and assess '
    'internalization or other modality-specific requirements. '
    'These are **proposed studies**, not experiments performed here.',

    '',

    '## 8. Reproducibility and data provenance',

    '',

    'The project retains a **948-gene** full-universe audit, '
    'a **15-gene** research panel, an experimental roadmap, '
    'and provenance JSON files recording source file hashes. '
    'The figures are linked to a three-row figure-source manifest '
    'and independent PNG checksums. The Colab working dataset '
    'also contains prior source-specific methods, model inventories, '
    'mapping audits, and earlier phase outputs. Raw public-source '
    'molecular data are not packaged with the compact research '
    'checkpoints and must be obtained in compliance with their '
    'original access and reuse conditions.',

    '',

    '**Key local research outputs:**',

    '',

    '- `results/tables/surfacetarget_phase5_full_universe_panel_audit.csv`',

    '- `results/tables/surfacetarget_phase5_research_panel_v1.csv`',

    '- `results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv`',

    '- `results/tables/surfacetarget_phase6_figure_manifest.csv`',

    '- `data/metadata/surfacetarget_phase5_research_panel_provenance.json`',

    '- `data/metadata/surfacetarget_phase6_figure_provenance.json`',

    '',

    '## 9. Conclusion',

    '',

    'SurfaceTarget-CRC narrows a 948-gene transcriptomic research '
    'universe into biologically distinct review lanes, audits '
    'orthogonal proteomic, topology, normal-tissue, and functional '
    'evidence, and documents a deliberately selected 15-gene panel. '
    'The main outcome is a **traceable experimental validation '
    'roadmap**, not proof of a clinically actionable target. '
    'Tissue selectivity and live-cell accessibility are the '
    'primary evidence gaps before any treatment-oriented '
    'claims can be made.',

    '',

    '---',

    '',

    '**Publication readiness note:** This draft is grounded in '
    'the validated Phase 5–6 tables. Before external submission, '
    'cross-check the earlier Phase 1–4 sample-selection and '
    'statistical-method details against their original notebooks/'
    'provenance and add formal dataset version citations, '
    'accession identifiers, and literature references.',

    ''

)


# ============================================================
# 12. SAVE FORMAL RESEARCH REPORT
# ============================================================

report_path = (
    REPORTS /
    'SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md'
)


report_path.write_text(
    '\n'.join(report),
    encoding='utf-8'
)


# ============================================================
# 13. PREPARE SOURCE-TO-CLAIM TRACEABILITY
# ============================================================

print('\nGENERATING RESEARCH CLAIM AUDIT')


claims = [

    (
        'candidate_universe',
        948,
        FULL_PATH,
        'Full candidate-gene table'
    ),

    (
        'panel_size',
        15,
        PANEL_PATH,
        'Research panel'
    ),

    (
        'cptac_enriched_candidates',
        coverage['Qualifying CPTAC enrichment'],
        FULL_PATH,
        'protein_enriched'
    ),

    (
        'stronger_surface_candidates',
        coverage['Stronger surface-related annotation'],
        FULL_PATH,
        'phase5_stronger_surface_annotation'
    ),

    (
        'hpa_normal_ihc_candidates',
        coverage['HPA normal-tissue IHC'],
        FULL_PATH,
        'phase5_normal_ihc_available'
    ),

    (
        'depmap_mapped_candidates',
        coverage['Verified DepMap mappings'],
        FULL_PATH,
        'depmap_final_mapping_available'
    ),

    (
        'depmap_high_coverage_candidates',
        coverage[
            'High-coverage DepMap genes (>=65/81 models)'
        ],
        FULL_PATH,
        'depmap_crc_models_measured'
    ),

    (
        'surface_review_candidates',
        len(surface),
        FULL_PATH,
        'phase5_followup_lane'
    ),

    (
        'three_way_top20_candidates',
        len(all_three),
        FULL_PATH,
        'Three rank fields'
    )

]


claims_frame = pd.DataFrame([

    {
        'claim': name,

        'validated_value': val,

        'source_path': str(source),

        'source_sha256': sha256_file(source),

        'field_or_method': field
    }

    for name, val, source, field in claims

])


claims_path = (
    TABLES /
    'surfacetarget_phase6_report_claim_audit.csv'
)


claims_frame.to_csv(
    claims_path,
    index=False
)


# ============================================================
# 14. CREATE COMPACT REPORT EVIDENCE TABLE
# ============================================================

compact_fields = [

    'gene_id',

    'gene_name',

    'phase56_review_group',

    'phase5_rna_conservative_log2fc',

    'phase5_protein_evidence_state',

    'phase5_topology_class',

    'phase55_normal_tissues_medium_high',

    'depmap_crc_median_gene_effect'

]


panel_table_path = (
    TABLES /
    'surfacetarget_phase6_report_panel_summary.csv'
)


panel[compact_fields].to_csv(
    panel_table_path,
    index=False
)


# ============================================================
# 15. VALIDATE REPORT OUTPUTS
# ============================================================

require(
    report_path.stat().st_size > 5000,
    'Report unexpectedly short'
)


require(
    len(pd.read_csv(claims_path)) == len(claims),
    'Claim audit row mismatch'
)


require(
    len(pd.read_csv(panel_table_path)) == 15,
    'Report panel table row mismatch'
)


# ============================================================
# 16. SAVE REPORT PROVENANCE
# ============================================================

provenance_path = (
    META /
    'surfacetarget_phase6_report_draft_provenance.json'
)


provenance = {

    'project': 'SurfaceTarget-CRC',

    'patch': '6.2',

    'generated_at_utc':
        datetime.now(timezone.utc).isoformat(),

    'report_path':
        str(report_path),

    'report_sha256':
        sha256_file(report_path),

    'report_claim_audit':
        str(claims_path),

    'report_claim_audit_sha256':
        sha256_file(claims_path),

    'report_panel_summary':
        str(panel_table_path),

    'report_panel_summary_sha256':
        sha256_file(panel_table_path),

    'sources_sha256': {

        name: sha256_file(path)
        for name, path in files.items()

    },

    'figure_checksums_verified': True,

    'panel_count': len(panel),

    'full_candidate_count': len(full),

    'scenario_comparison_performed': True,

    'new_target_ranking': False,

    'clinical_safety_inferred': False,

    'experimental_validation_performed': False,

    'outstanding_before_publication': [

        'Confirm detailed Phase 1–4 cohort methods and '
        'thresholds from original notebooks and provenance',

        'Add formal source accessions, version identifiers, '
        'and bibliographic references',

        'Review narrative claims with a domain expert',

        'Inspect report figures at final export resolution',

        'Confirm permissions and licensing before '
        'publishing data or derived files'

    ]

}


provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding='utf-8'
)


print('\nReport saved:', report_path)

print(
    'Report words:',
    len(report_path.read_text(encoding='utf-8').split())
)

print('Claim audit:', claims_path)

print('Panel summary:', panel_table_path)

print('Provenance:', provenance_path)


# ============================================================
# 17. CREATE PHASE 6 CHECKPOINT
# ============================================================

print('\nCREATING RESEARCH CHECKPOINT')


stamp = (
    datetime.now(timezone.utc)
    .strftime('%Y%m%d_%H%M%S')
)


checkpoint_path = (
    CHECKPOINTS /
    f'SurfaceTarget_CRC_Phase6_Patch62_{stamp}.zip'
)


backup_folders = [

    'data/metadata',

    'data/processed',

    'data/raw/cptac_colon',

    'data/raw/hpa',

    'data/raw/uniprot',

    'data/raw/hgnc',

    'results/tables',

    'results/figures',

    'src',

    'reports',

    'logs'

]


with ZipFile(
    checkpoint_path,
    'w',
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if source_folder.is_dir():

            for file_path in source_folder.rglob('*'):

                if file_path.is_file():

                    archive.write(
                        file_path,
                        arcname=str(
                            file_path.relative_to(ROOT)
                        )
                    )


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print('\n' + '=' * 68)

print('PATCH 6.2 COMPLETE')

print(
    'Report sections: Abstract, Background, Methods, Results, '
    'Candidate interpretation,'
)

print(
    'Limitations, Experimental follow-up, '
    'Reproducibility, Conclusion'
)

print('Source data changed: NO')

print('All 948 genes preserved; 15 panel genes retained')

print('Checkpoint:', checkpoint_path)

print(
    f'Checkpoint size: '
    f'{checkpoint_path.stat().st_size / 1024**2:.2f} MiB'
)

print('=' * 68)